# Reward Weighting Experiments — Pareto Frontier Analysis

This notebook trains one REINFORCE agent per reward configuration and evaluates each on a held-out test year.  
The three objectives are:

| Objective | Definition | Direction |
|---|---|---|
| **Accuracy** | Fraction of inspections that yielded a real violation | maximize |
| **Throughput** | Total reports closed during the test year | maximize |
| **Fairness** | `1 − borough_equity_score` (TV distance from proportional coverage) | maximize |

After all runs are complete the notebook plots the Pareto frontier in each pair of objective dimensions and a 3-D scatter.

## 1  Imports & shared environment settings

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401  (registers 3-D projection)
from itertools import combinations

from housing_env import HousingEnv, RewardWeights
from train_reinforce import (
    train_and_evaluate,
    evaluate_agent,
    evaluate_random,
    flatten_obs,
    report_mask_from,
)

In [ ]:
# ── Shared settings ────────────────────────────────────────────────────────────
DATA_PATH          = "data/311_preproc.csv"
NUM_INSPECTORS     = 150
MAX_ACTIVE_REPORTS = 600
TRAIN_YEARS        = 1      # keep short so each run finishes in ~2-4 min
TEST_YEARS         = 1
NUM_EPISODES       = 20     # increase to 50+ for production-quality results
SEED               = 42

TRAIN_START = "2024-01-01"
TEST_START  = "2025-01-01"

print(f"Train window : {TRAIN_START}  ({TRAIN_YEARS} yr)")
print(f"Test  window : {TEST_START}   ({TEST_YEARS} yr)")

## 2  Reward configurations

Each entry is a `(label, RewardWeights)` pair.  
The decomposition inside the environment is:

```
step_reward =  w_accuracy  × 40   (only for VIOLATION outcome)
            +  w_throughput × {VIOLATION:10, DUPLICATE:20, others:10}
            +  w_fairness   × (expected_borough_share − actual_share) × 10
             − open_penalty × n_open_reports
```

Default weights (`1, 1, 0, 0`) reproduce the original `50 / 20 / 10` structure exactly.

In [ ]:
CONFIGS = [
    # (label, RewardWeights)
    ("Baseline",           RewardWeights(w_accuracy=1.0, w_throughput=1.0, w_fairness=0.0, open_penalty=0.00)),
    ("High Accuracy",      RewardWeights(w_accuracy=4.0, w_throughput=0.5, w_fairness=0.0, open_penalty=0.00)),
    ("High Throughput",    RewardWeights(w_accuracy=0.0, w_throughput=2.0, w_fairness=0.0, open_penalty=0.00)),
    ("High Fairness",      RewardWeights(w_accuracy=1.0, w_throughput=1.0, w_fairness=3.0, open_penalty=0.00)),
    ("Backlog Penalty",    RewardWeights(w_accuracy=1.0, w_throughput=1.0, w_fairness=0.0, open_penalty=0.05)),
    ("Accuracy+Fairness",  RewardWeights(w_accuracy=3.0, w_throughput=0.5, w_fairness=2.0, open_penalty=0.00)),
    ("Throughput+Fairness",RewardWeights(w_accuracy=0.0, w_throughput=2.0, w_fairness=2.0, open_penalty=0.00)),
]

print(f"{len(CONFIGS)} configurations defined.")
for label, rw in CONFIGS:
    print(f"  {label:<22s}  accuracy={rw.w_accuracy:.1f}  throughput={rw.w_throughput:.1f}  "
          f"fairness={rw.w_fairness:.1f}  open_penalty={rw.open_penalty:.3f}")

## 3  Training loop

Each configuration is trained independently.  All results are stored in `all_results`.

In [ ]:
import os, json

RESULTS_DIR = "results/reward_experiments"

def _label_to_slug(label: str) -> str:
    return label.lower().replace(" ", "_").replace("+", "_").replace("/", "_")


def save_run_results(label: str, result: dict, output_dir: str = RESULTS_DIR) -> str:
    """Save all data from one train_and_evaluate run to *output_dir/{slug}/*.

    Files written
    -------------
    eval_reinforce.csv   per-step evaluation data for the REINFORCE agent
    eval_random.csv      per-step evaluation data for the random baseline
    training_history.csv per-episode training metrics
    meta.json            reward weights, elapsed time, open_reports, comparison
    """
    slug = _label_to_slug(label)
    run_dir = os.path.join(output_dir, slug)
    os.makedirs(run_dir, exist_ok=True)

    def _eval_df(ev: dict) -> pd.DataFrame:
        n = len(ev["rewards"])
        return pd.DataFrame({
            "step":                      range(n),
            "reward":                    ev["rewards"],
            "cumulative_reward":         list(np.cumsum(ev["rewards"])),
            "violations":                ev["violations"],
            "cumulative_violations":     list(np.cumsum(ev["violations"])),
            "reports_closed":            ev["reports_closed"],
            "cumulative_reports_closed": list(np.cumsum(ev["reports_closed"])),
        })

    _eval_df(result["reinforce_eval"]).to_csv(
        os.path.join(run_dir, "eval_reinforce.csv"), index=False)

    _eval_df(result["random_eval"]).to_csv(
        os.path.join(run_dir, "eval_random.csv"), index=False)

    pd.DataFrame(result["training_history"]).to_csv(
        os.path.join(run_dir, "training_history.csv"), index=False)

    rw = result["reward_weights"]
    meta = {
        "label":        label,
        "train_start":  TRAIN_START,
        "test_start":   TEST_START,
        "train_years":  TRAIN_YEARS,
        "test_years":   TEST_YEARS,
        "num_episodes": NUM_EPISODES,
        "elapsed_s":    round(result.get("elapsed", 0), 1),
        "reward_weights": {
            "w_accuracy":   rw.w_accuracy,
            "w_throughput": rw.w_throughput,
            "w_fairness":   rw.w_fairness,
            "open_penalty": rw.open_penalty,
        },
        "open_reports_reinforce": result["reinforce_eval"]["open_reports"],
        "open_reports_random":    result["random_eval"]["open_reports"],
        "comparison":             result.get("comparison", {}),
    }
    with open(os.path.join(run_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"  Saved \u2192 {run_dir}/")
    return run_dir


def load_run_results(output_dir: str = RESULTS_DIR) -> dict:
    """Load all previously saved runs from *output_dir*.

    Returns
    -------
    dict[label, dict]  keyed by the original config label stored in meta.json.
    Each value contains:
        eval_reinforce    pd.DataFrame  (per-step)
        eval_random       pd.DataFrame  (per-step)
        training_history  pd.DataFrame  (per-episode)
        meta              dict
    """
    loaded = {}
    if not os.path.isdir(output_dir):
        print(f"Directory not found: {output_dir}")
        return loaded

    for slug in sorted(os.listdir(output_dir)):
        run_dir = os.path.join(output_dir, slug)
        meta_path = os.path.join(run_dir, "meta.json")
        if not os.path.isfile(meta_path):
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        label = meta["label"]
        loaded[label] = {
            "eval_reinforce":   pd.read_csv(os.path.join(run_dir, "eval_reinforce.csv")),
            "eval_random":      pd.read_csv(os.path.join(run_dir, "eval_random.csv")),
            "training_history": pd.read_csv(os.path.join(run_dir, "training_history.csv")),
            "meta":             meta,
        }
        print(f"Loaded '{label}' from {run_dir}/")

    return loaded


print(f"Results will be saved to: {RESULTS_DIR}/")


In [ ]:
all_results = {}   # label → dict returned by train_and_evaluate

for label, rw in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Config: {label}")
    print(f"  Weights: accuracy={rw.w_accuracy} throughput={rw.w_throughput} "
          f"fairness={rw.w_fairness} open_penalty={rw.open_penalty}")
    print(f"{'='*60}")
    t0 = time.time()

    result = train_and_evaluate(
        data_path=DATA_PATH,
        num_inspectors=NUM_INSPECTORS,
        max_active_reports=MAX_ACTIVE_REPORTS,
        train_years=TRAIN_YEARS,
        test_years=TEST_YEARS,
        train_start_date=TRAIN_START,
        test_start_date=TEST_START,
        num_episodes=NUM_EPISODES,
        seed=SEED,
        reward_weights=rw,
        save_path=None,
        plot=False,
        verbose=True,
    )
    result["reward_weights"] = rw
    result["elapsed"] = time.time() - t0
    all_results[label] = result

    save_run_results(label, result)

    print(f"  Done in {result['elapsed']:.0f}s")

print("\nAll configurations trained.")


In [ ]:
# ── Load saved results (run this cell independently to skip re-training) ──────
# saved_runs = load_run_results()
#
# Each entry: saved_runs[label]["eval_reinforce"]   -> per-step DataFrame
#             saved_runs[label]["eval_random"]       -> per-step DataFrame
#             saved_runs[label]["training_history"]  -> per-episode DataFrame
#             saved_runs[label]["meta"]              -> dict (weights, dates, etc.)
#
# Example: plot cumulative reward for every config
#
# for label, run in saved_runs.items():
#     df = run["eval_reinforce"]
#     plt.plot(df["cumulative_reward"], label=label)
# plt.legend(); plt.show()

saved_runs = load_run_results()
print(f"\n{len(saved_runs)} run(s) loaded.")


## 4  Extract objective metrics

All three objectives are extracted from the **test-year evaluation** of each REINFORCE agent.

In [ ]:
def extract_metrics(label: str, result: dict) -> dict:
    """Compute scalar objectives from a train_and_evaluate result dict."""
    ev = result["reinforce_eval"]

    total_inspections = sum(ev["reports_closed"])   # reports_closed = inspections performed
    total_violations  = sum(ev["violations"])

    # Objective 1: accuracy — fraction of inspections that found a violation
    accuracy = total_violations / max(1, total_inspections)

    # Objective 2: throughput — total reports closed (absolute count)
    throughput = total_inspections

    # Objective 3: fairness — mean of step-level (1 - equity_score)
    # borough_equity_score is the TV distance: 0 = fair, 1 = maximally skewed
    # We approximate it from the final borough_inspections in the last eval step.
    # train_and_evaluate doesn't return per-step equity scores, so we re-compute
    # using the stored borough_inspections from the last episode state via the
    # env's borough_equity_score() method.  We proxy it here from violations:
    #   if no fairness data is available we fall back to 0 (neutral).
    fairness = 1.0 - result.get("_final_equity_score", 0.0)

    return {
        "label":      label,
        "accuracy":   accuracy,
        "throughput": throughput,
        "fairness":   fairness,
        "violations": total_violations,
        "w_accuracy":   result["reward_weights"].w_accuracy,
        "w_throughput": result["reward_weights"].w_throughput,
        "w_fairness":   result["reward_weights"].w_fairness,
        "open_penalty": result["reward_weights"].open_penalty,
    }


# Re-run evaluation to capture per-step borough equity scores
print("Re-evaluating agents to collect borough equity scores...")

EVAL_MAX_STEPS = int(TEST_YEARS * 365 * 4)  # upper bound; env terminates at test_years

for label, result in all_results.items():
    env = HousingEnv(
        num_inspectors=NUM_INSPECTORS,
        max_active_reports=MAX_ACTIVE_REPORTS,
        years=TEST_YEARS,
        hierarchical=True,
        data_path=DATA_PATH,
        start_date_str=TEST_START,
        reward_weights=result["reward_weights"],
    )
    obs, _ = env.reset()
    agent   = result["agent"]
    equity_scores = []

    for _ in range(EVAL_MAX_STEPS):
        rmask = report_mask_from(obs)
        action = agent.predict(flatten_obs(obs), rmask) if rmask.any() else 0
        obs, _, terminated, truncated, info = env.step(action)
        equity_scores.append(info["borough_equity_score"])
        if terminated or truncated:
            break

    result["_equity_scores"]        = equity_scores
    result["_final_equity_score"]   = float(np.mean(equity_scores))
    env.close()
    print(f"  {label:<22s}  mean equity score = {result['_final_equity_score']:.4f}")

print("Done.")

In [ ]:
# Build summary DataFrame
metrics_list = [extract_metrics(label, result) for label, result in all_results.items()]
df_metrics   = pd.DataFrame(metrics_list).set_index("label")

display(df_metrics[["accuracy", "throughput", "fairness", "violations",
                     "w_accuracy", "w_throughput", "w_fairness", "open_penalty"]]
        .style
        .format({
            "accuracy":    "{:.4f}",
            "throughput":  "{:.0f}",
            "fairness":    "{:.4f}",
            "violations":  "{:.0f}",
        })
        .background_gradient(subset=["accuracy", "throughput", "fairness"], cmap="RdYlGn"))

## 5  Training curves

In [ ]:
n_configs = len(all_results)
fig, axes = plt.subplots(1, n_configs, figsize=(4 * n_configs, 4), sharey=False)
fig.suptitle("Training Curves by Reward Configuration", fontsize=13, y=1.02)

for ax, (label, result) in zip(axes, all_results.items()):
    ep_rewards = [h["total_reward"] for h in result["training_history"]]
    window = max(1, len(ep_rewards) // 5)
    rolling = pd.Series(ep_rewards).rolling(window, min_periods=1).mean()
    ax.plot(ep_rewards, alpha=0.3, color="steelblue")
    ax.plot(rolling, color="steelblue", linewidth=2)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total Reward")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_experiment_training_curves.png", dpi=130, bbox_inches="tight")
plt.show()

## 6  Objective bar chart

In [ ]:
objectives = ["accuracy", "throughput", "fairness"]
labels_list = list(all_results.keys())
x = np.arange(len(labels_list))
colors = plt.cm.tab10(np.linspace(0, 1, len(labels_list)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Objective Metrics per Reward Configuration", fontsize=13)

titles = {
    "accuracy":   "Accuracy\n(violations / inspections)",
    "throughput": "Throughput\n(total reports closed)",
    "fairness":   "Fairness\n(1 − mean borough equity score)",
}

for ax, obj in zip(axes, objectives):
    vals = df_metrics[obj].values
    bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_list, rotation=35, ha="right", fontsize=8)
    ax.set_title(titles[obj], fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")
    # annotate bar tops
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f"{val:.3f}" if obj != "throughput" else f"{val:.0f}",
                ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("reward_experiment_bar_chart.png", dpi=130, bbox_inches="tight")
plt.show()

## 7  Pareto frontier

A point is **Pareto-optimal** if no other configuration dominates it on every objective simultaneously.  
Points on the frontier represent genuine tradeoffs — you cannot improve one objective without sacrificing another.

In [ ]:
def is_pareto_optimal(scores: np.ndarray) -> np.ndarray:
    """Return boolean array (n_points,); True = Pareto-optimal.
    
    All objectives are assumed to be maximized.
    Point i is dominated if there exists j such that
        scores[j] >= scores[i] on all objectives
    AND scores[j] > scores[i] on at least one objective.
    """
    n = len(scores)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(scores[j] >= scores[i]) and np.any(scores[j] > scores[i]):
                is_pareto[i] = False
                break
    return is_pareto


score_matrix = df_metrics[["accuracy", "throughput", "fairness"]].values.copy()

# Normalise each column to [0, 1] so scatter axes are comparable
score_norm = (score_matrix - score_matrix.min(axis=0)) / (
    score_matrix.ptp(axis=0) + 1e-12
)

pareto_mask = is_pareto_optimal(score_matrix)
pareto_labels = df_metrics.index[pareto_mask].tolist()

print("Pareto-optimal configurations:")
for lbl in pareto_labels:
    row = df_metrics.loc[lbl]
    print(f"  {lbl:<22s}  accuracy={row.accuracy:.4f}  "
          f"throughput={row.throughput:.0f}  fairness={row.fairness:.4f}")

In [ ]:
# ── 2-D Pareto plots for every pair of objectives ────────────────────────────
obj_pairs = list(combinations(range(3), 2))
obj_names = ["Accuracy", "Throughput (norm.)", "Fairness"]
obj_keys  = ["accuracy", "throughput", "fairness"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Pairwise Pareto Frontiers  (★ = Pareto-optimal)", fontsize=13)

for ax, (i, j) in zip(axes, obj_pairs):
    xi, xj = score_norm[:, i], score_norm[:, j]

    # All points
    ax.scatter(xi[~pareto_mask], xj[~pareto_mask],
               s=80, color="steelblue", alpha=0.6, zorder=2, label="Dominated")
    ax.scatter(xi[pareto_mask],  xj[pareto_mask],
               s=160, marker="*", color="crimson", zorder=3, label="Pareto-optimal")

    # Draw a step-function frontier on the Pareto points
    if pareto_mask.sum() > 1:
        px = xi[pareto_mask]
        py = xj[pareto_mask]
        order = np.argsort(px)
        ax.step(px[order], py[order], where="post",
                color="crimson", linewidth=1.4, linestyle="--", alpha=0.7, zorder=1)

    # Labels
    for k, lbl in enumerate(df_metrics.index):
        ax.annotate(
            lbl, (xi[k], xj[k]),
            textcoords="offset points", xytext=(5, 4),
            fontsize=7, color="black",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")],
        )

    ax.set_xlabel(obj_names[i], fontsize=10)
    ax.set_ylabel(obj_names[j], fontsize=10)
    ax.set_title(f"{obj_names[i]} vs {obj_names[j]}", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_experiment_pareto_2d.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ── 3-D scatter ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection="3d")

x3, y3, z3 = score_norm[:, 0], score_norm[:, 1], score_norm[:, 2]

ax.scatter(x3[~pareto_mask], y3[~pareto_mask], z3[~pareto_mask],
           s=60, c="steelblue", alpha=0.6, label="Dominated")
ax.scatter(x3[pareto_mask],  y3[pareto_mask],  z3[pareto_mask],
           s=180, c="crimson", marker="*", depthshade=False, label="Pareto-optimal")

for k, lbl in enumerate(df_metrics.index):
    ax.text(x3[k], y3[k], z3[k], f"  {lbl}", fontsize=7, zorder=5)

ax.set_xlabel("Accuracy (norm.)", labelpad=8)
ax.set_ylabel("Throughput (norm.)", labelpad=8)
ax.set_zlabel("Fairness (norm.)", labelpad=8)
ax.set_title("3-D Objective Space  (★ = Pareto-optimal)", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("reward_experiment_pareto_3d.png", dpi=130, bbox_inches="tight")
plt.show()

## 8  Borough equity over time

For each configuration, plot how the per-step `borough_equity_score` (TV distance) evolves during the test year.  
A score of 0 means inspections are perfectly proportional to open reports across boroughs.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors_cycle = plt.cm.tab10(np.linspace(0, 1, len(all_results)))

for (label, result), color in zip(all_results.items(), colors_cycle):
    scores = result.get("_equity_scores", [])
    if not scores:
        continue
    window = max(1, len(scores) // 20)
    smoothed = pd.Series(scores).rolling(window, min_periods=1).mean()
    lw = 2.5 if label in pareto_labels else 1.0
    ls = "-" if label in pareto_labels else "--"
    ax.plot(smoothed, label=label, color=color, linewidth=lw, linestyle=ls)

ax.set_xlabel("Evaluation Timestep")
ax.set_ylabel("Borough Equity Score (TV distance)")
ax.set_title("Borough Equity Score Over Test Year\n(lower = more equitable; solid = Pareto-optimal)", fontsize=11)
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_experiment_equity_over_time.png", dpi=130, bbox_inches="tight")
plt.show()

## 9  Parallel coordinates — all objectives simultaneously

In [ ]:
from pandas.plotting import parallel_coordinates

pc_df = df_metrics[["accuracy", "throughput", "fairness"]].copy()

# Normalise each column so axes are comparable
for col in ["accuracy", "throughput", "fairness"]:
    lo, hi = pc_df[col].min(), pc_df[col].max()
    pc_df[col] = (pc_df[col] - lo) / (hi - lo + 1e-12)

pc_df["pareto"] = ["Pareto-optimal" if lbl in pareto_labels else "Dominated"
                   for lbl in pc_df.index]
pc_df["config"] = pc_df.index

fig, ax = plt.subplots(figsize=(10, 5))
parallel_coordinates(
    pc_df.reset_index(drop=True),
    class_column="pareto",
    cols=["accuracy", "throughput", "fairness"],
    color=["crimson", "steelblue"],
    linewidth=2,
    alpha=0.75,
    ax=ax,
)

# Annotate each line with its config label
for i, row in pc_df.reset_index().iterrows():
    ax.annotate(row["label"], xy=(2, row["fairness"]),
                xytext=(2.05, row["fairness"]),
                fontsize=7, va="center",
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])

ax.set_title("Parallel Coordinates — Normalised Objectives", fontsize=12)
ax.set_ylabel("Normalised Score (0 = worst, 1 = best)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reward_experiment_parallel_coords.png", dpi=130, bbox_inches="tight")
plt.show()

## 10  Summary

In [ ]:
print("=" * 65)
print("REWARD WEIGHTING EXPERIMENT SUMMARY")
print("=" * 65)
print(f"  Training  : {TRAIN_YEARS} year(s) × {NUM_EPISODES} episodes per config")
print(f"  Evaluation: {TEST_YEARS} held-out test year")
print()

summary = df_metrics[["accuracy", "throughput", "fairness"]].copy()
summary["pareto_optimal"] = [lbl in pareto_labels for lbl in summary.index]

print(summary.sort_values("throughput", ascending=False).to_string())

print()
print("Pareto-optimal configs:", ", ".join(pareto_labels))
print()
print("Key tradeoffs:")
best_acc  = df_metrics["accuracy"].idxmax()
best_tput = df_metrics["throughput"].idxmax()
best_fair = df_metrics["fairness"].idxmax()
print(f"  Highest accuracy   : {best_acc}  ({df_metrics.loc[best_acc, 'accuracy']:.4f})")
print(f"  Highest throughput : {best_tput}  ({df_metrics.loc[best_tput, 'throughput']:.0f} reports closed)")
print(f"  Highest fairness   : {best_fair}  ({df_metrics.loc[best_fair, 'fairness']:.4f})")